# **Proyecto Etapa 3 — Aprendizaje Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

**Actividad individual**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A01139580 |

---
## 1. Introducción: Aprendizaje Supervisado

### 1.1 Concepto general

El **aprendizaje supervisado** es una rama del aprendizaje automático en la que un modelo es entrenado a partir de un conjunto de pares entrada–salida etiquetados $(x_i, y_i)$. El objetivo es aprender una función $f : X \rightarrow Y$ que generalice bien a datos no vistos, minimizando alguna función de pérdida sobre el conjunto de entrenamiento.

Formalmente, dado un conjunto de entrenamiento $\mathcal{D}_{\text{train}} = \{(x_1, y_1), \ldots, (x_n, y_n)\}$, el modelo aprende los parámetros $\theta^*$ tal que:

$$\theta^* = \arg\min_\theta \frac{1}{n} \sum_{i=1}^{n} \mathcal{L}(f_\theta(x_i),\, y_i)$$

donde $\mathcal{L}$ es la función de pérdida adecuada al problema (entropía cruzada para clasificación, error cuadrático para regresión).

Los dos grandes tipos de tareas supervisadas son:
- **Clasificación**: $Y$ es un conjunto discreto de clases (e.g., predecir el tipo de tejido).
- **Regresión**: $Y \subseteq \mathbb{R}$ (e.g., predecir un valor continuo de expresión génica).

---

### 1.2 Algoritmos representativos en la literatura

| Algoritmo | Tipo | Fortalezas | Limitaciones |
|-----------|------|------------|--------------|
| **Árbol de Decisión** (Decision Tree) | Clasificación / Regresión | Interpretable, no requiere normalización | Sobreajuste fácil; inestable ante pequeñas variaciones |
| **Random Forest** | Clasificación / Regresión | Robusto al sobreajuste; maneja alta dimensionalidad | Menos interpretable; costoso en memoria |
| **Gradient Boosted Trees (GBT)** | Clasificación / Regresión | Alta precisión; captura interacciones no lineales | Entrenamiento secuencial; más lento que RF |
| **Regresión Logística** | Clasificación | Simple, interpretable, probabilístico | Supone linealidad; sensible a multicolinealidad |
| **SVM (Support Vector Machine)** | Clasificación / Regresión | Efectivo en alta dimensión; margen máximo | Difícil de escalar a millones de instancias |
| **Perceptrón Multicapa (MLP)** | Clasificación / Regresión | Captura relaciones muy complejas | Requiere mucho dato y tunning cuidadoso |
| **Naive Bayes** | Clasificación | Muy rápido; bien calibrado con poco dato | Asume independencia entre features |

---

### 1.3 Algoritmos disponibles en PySpark MLlib

PySpark MLlib (`pyspark.ml.classification`) ofrece implementaciones distribuidas de los siguientes algoritmos de clasificación:

| Clase PySpark | Algoritmo |
|---------------|-----------|
| `DecisionTreeClassifier` | Árbol de decisión |
| `RandomForestClassifier` | Random Forest |
| `GBTClassifier` | Gradient Boosted Trees |
| `LogisticRegression` | Regresión logística (multinomial) |
| `MultilayerPerceptronClassifier` | Red neuronal MLP |
| `LinearSVC` | SVM lineal |
| `NaiveBayes` | Naive Bayes |
| `FMClassifier` | Factorization Machines |

El pipeline de MLlib sigue el patrón `Transformer → Estimator → Model`, compatible con `Pipeline` y `CrossValidator` para validación cruzada distribuida.

---

### 1.4 Algoritmo seleccionado: Random Forest

Se selecciona **Random Forest** por las siguientes razones aplicadas al contexto GTEx:

1. **Alta dimensionalidad**: cada muestra tiene miles de features (genes). RF selecciona aleatoriamente un subconjunto de features en cada split, lo que reduce la varianza sin requerir selección manual de genes.
2. **Robustez al sobreajuste**: el promedio de múltiples árboles decorrelacionados mitiga el sobreajuste que sufriría un árbol individual sobre datos de expresión génica ruidosos.
3. **Importancia de variables**: RF produce un ranking de importancia de genes (`featureImportances`), útil para interpretar qué genes distinguen los grupos de tejido.
4. **No requiere normalización estricta**: los valores TPM ya son comparables entre muestras, pero RF es insensible a escalas, a diferencia de MLP o SVM.

---
## 2. Selección de los datos

### 2.1 Estrategia

La muestra M del equipo (Etapa 2) abarca las 10 particiones P01–P10 construidas a partir del muestreo estratificado proporcional sobre el dataset GTEx V10. Para esta actividad individual se construye una sub-muestra **M'** con las siguientes restricciones:

- Se trabaja con las **particiones cardiovasculares y musculoesqueléticas** (P05, P06, P07, P08) para mantener el problema manejable.
- Se aplica un muestreo aleatorio adicional de **200 muestras por partición** (800 muestras totales), suficiente para entrenar un clasificador binario de sexo sin tiempos de procesamiento excesivos.
- La variable objetivo es **SEX_LABEL** (Masculino / Femenino), una tarea de clasificación binaria.
- Las features son los valores de expresión TPM de **500 genes** seleccionados aleatoriamente (columna por gen).

**Justificación de la variable objetivo:** predecir el sexo biológico a partir de la expresión génica es una tarea con respuesta conocida en la literatura — los genes del cromosoma Y (RPS4Y1, KDM5D, EIF1AY, etc.) son altamente expresados en tejidos masculinos y ausentes en femeninos. Esto permite verificar que el modelo aprende señal biológica real, no artefactos numéricos.

In [1]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Set JAVA_HOME to the JDK bundled with the big-data conda env.
# On Windows, sys.executable is <env_root>\python.exe so dirname gives the env root.
_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sub-sample parameters
TARGET_TISSUE_GROUPS = ['Cardiovascular', 'Musculoesqueletico']
SAMPLES_PER_PARTITION = 200   # max samples taken from each of the 4 partitions
N_GENES_SUBSAMPLE     = 500   # number of randomly selected gene columns
GENE_STEP             = 100   # read every Nth column from the raw file first

print(f'Semilla aleatoria : {RANDOM_SEED}')
print(f'Grupos de tejido  : {TARGET_TISSUE_GROUPS}')
print(f'Muestras/partición: {SAMPLES_PER_PARTITION}')
print(f'Genes seleccionados: {N_GENES_SUBSAMPLE}')
print(f'JAVA_HOME seteado : {os.environ.get("JAVA_HOME", "ERROR - no seteado")}')

Semilla aleatoria : 42
Grupos de tejido  : ['Cardiovascular', 'Musculoesqueletico']
Muestras/partición: 200
Genes seleccionados: 500
JAVA_HOME seteado : C:\Users\diego\anaconda3\envs\big-data\Library\lib\jvm


In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_SupervisedLearning_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark

In [3]:
# --- Load metadata ---
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

# Tissue group mapping (same as Etapa 2)
tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

# Filter to the two target tissue groups (P05, P06, P07, P08)
meta_target = meta_df.filter(F.col('TISSUE_GROUP').isin(TARGET_TISSUE_GROUPS))

print('Distribución por partición (target):')
meta_target.groupBy('TISSUE_GROUP', 'SEX_LABEL').count().orderBy('TISSUE_GROUP', 'SEX_LABEL').show()

Distribución por partición (target):


+------------------+---------+-----+
|      TISSUE_GROUP|SEX_LABEL|count|
+------------------+---------+-----+
|    Cardiovascular| Femenino|  771|
|    Cardiovascular|Masculino| 1573|
|Musculoesqueletico| Femenino| 1349|
|Musculoesqueletico|Masculino| 2827|
+------------------+---------+-----+



In [4]:
# --- Build M': stratified sub-sample, up to SAMPLES_PER_PARTITION per partition ---
# Collect sample IDs per (TISSUE_GROUP, SEX_LABEL) partition
from collections import defaultdict

meta_rows = meta_target.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD').collect()

# Group by partition
partition_samples = defaultdict(list)
for row in meta_rows:
    key = (row['TISSUE_GROUP'], row['SEX_LABEL'])
    partition_samples[key].append((row['COL_NAME'], row['SMTSD']))

# Stratified sampling within each partition: proportional to SMTSD subtypes
rng = random.Random(RANDOM_SEED)
selected_col_names = []
selected_meta = []   # [(col_name, tissue_group, sex_label)]

for (tg, sx), items in sorted(partition_samples.items()):
    n_partition = len(items)
    n_select = min(SAMPLES_PER_PARTITION, n_partition)

    # Group by SMTSD
    by_subtype = defaultdict(list)
    for col, smtsd in items:
        by_subtype[smtsd].append(col)

    # Proportional allocation
    sampled = []
    for smtsd, cols in by_subtype.items():
        n_strata = max(1, round(n_select * len(cols) / n_partition))
        n_strata = min(n_strata, len(cols))
        sampled.extend(rng.sample(cols, n_strata))

    # Trim to exact target if over-sampled due to rounding
    sampled = sampled[:n_select]

    for col in sampled:
        selected_col_names.append(col)
        selected_meta.append((col, tg, sx))

    print(f'  {tg:<22} + {sx:<12}: {len(sampled)} muestras seleccionadas de {n_partition}')

print(f'\nTotal M\' : {len(selected_col_names)} muestras')

  Cardiovascular         + Femenino    : 200 muestras seleccionadas de 771
  Cardiovascular         + Masculino   : 199 muestras seleccionadas de 1573
  Musculoesqueletico     + Femenino    : 200 muestras seleccionadas de 1349
  Musculoesqueletico     + Masculino   : 200 muestras seleccionadas de 2827

Total M' : 799 muestras


In [5]:
# --- Load TPM file columns for M' samples only ---
import pandas as pd
from pyspark.sql.functions import split as spark_split

# Read all column names
peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names_raw = peek.columns.tolist()
all_col_names_clean = [c.replace('-', '_').replace('.', '_') for c in all_col_names_raw]

# Map clean name -> original index
name_to_idx = {clean: idx for idx, clean in enumerate(all_col_names_clean)}

# Find indices for Name, Description + selected samples
fixed_cols = ['Name', 'Description']
fixed_indices = [name_to_idx[c] for c in fixed_cols]

# Keep only selected sample columns that exist in the file
valid_sample_cols = [c for c in selected_col_names if c in name_to_idx]
sample_indices = [name_to_idx[c] for c in valid_sample_cols]

all_selected_indices = fixed_indices + sample_indices
all_selected_names   = fixed_cols + valid_sample_cols

print(f'Columnas de muestra válidas: {len(valid_sample_cols)} / {len(selected_col_names)}')

# Load only gene rows with selected columns via Spark
raw_df = spark.read.text(FILE_PATH)
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))
split_col = spark_split(F.col('value'), '\t')

df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(all_selected_names[idx])
      for idx, i in enumerate(all_selected_indices)]
)

for c in valid_sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM M\': {df_tpm.count():,} genes x {len(valid_sample_cols)} muestras')
df_tpm.select(all_selected_names[:5]).show(3)

Columnas de muestra válidas: 799 / 799


DataFrame TPM M': 59,033 genes x 799 muestras


+-----------------+-----------+-----------------------+------------------------+------------------------+
|             Name|Description|GTEX_QDT8_0426_SM_32PKZ|GTEX_13CF3_2226_SM_5J2MX|GTEX_11GSP_2926_SM_5N9C2|
+-----------------+-----------+-----------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                    0.0|                     0.0|                     0.0|
|ENSG00000227232.5|     WASH7P|                5.78407|                 3.13248|                 3.17456|
|ENSG00000278267.1|  MIR6859-1|                    0.0|                     0.0|                     0.0|
+-----------------+-----------+-----------------------+------------------------+------------------------+
only showing top 3 rows


In [6]:
# --- Select N_GENES_SUBSAMPLE genes randomly to keep feature matrix manageable ---
gene_ids = [row['Name'] for row in df_tpm.select('Name').collect()]
selected_gene_ids = rng.sample(gene_ids, min(N_GENES_SUBSAMPLE, len(gene_ids)))
selected_gene_ids_set = set(selected_gene_ids)

df_tpm_sub = df_tpm.filter(F.col('Name').isin(selected_gene_ids_set))
print(f'Genes seleccionados: {df_tpm_sub.count()} de {len(gene_ids):,}')

Genes seleccionados: 500 de 59,033


In [7]:
# --- Pivot: rows=samples, columns=genes (transpose the TPM matrix) ---
# Collect the gene x sample matrix to pandas for pivoting (small enough: 500 genes x ~800 samples)
tpm_pd = df_tpm_sub.select(['Name'] + valid_sample_cols).toPandas()
tpm_pd = tpm_pd.set_index('Name')

# Transpose: rows=samples, columns=genes
tpm_T = tpm_pd.T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})

# Sanitize gene column names: GTEx IDs contain dots (e.g. ENSG00000253852.1)
# Spark interprets dots as nested field access, so replace with underscores.
rename_map = {g: g.replace('.', '_') for g in selected_gene_ids}
tpm_T = tpm_T.rename(columns=rename_map)
gene_cols = [rename_map[g] for g in selected_gene_ids]

# Attach labels from selected_meta
meta_dict = {col: (tg, sx) for col, tg, sx in selected_meta}
tpm_T['TISSUE_GROUP'] = tpm_T['COL_NAME'].map(lambda c: meta_dict.get(c, (None, None))[0])
tpm_T['SEX_LABEL']    = tpm_T['COL_NAME'].map(lambda c: meta_dict.get(c, (None, None))[1])
tpm_T = tpm_T.dropna(subset=['TISSUE_GROUP', 'SEX_LABEL'])

# Replace NaN TPM values with 0 (unexpressed genes)
tpm_T[gene_cols] = tpm_T[gene_cols].fillna(0.0)

print(f'Matriz M\' final: {tpm_T.shape[0]} muestras x {len(gene_cols)} features')
print('Distribución de clases:')
print(tpm_T['SEX_LABEL'].value_counts())

Matriz M' final: 799 muestras x 500 features
Distribución de clases:
SEX_LABEL
Femenino     400
Masculino    399
Name: count, dtype: int64


---
## 3. Preparación del conjunto de entrenamiento y prueba

### 3.1 Técnica de división

Se aplica una **división estratificada 80 / 20** (train / test):

- **80% entrenamiento**: proporciona suficiente dato para que Random Forest construya árboles robustos con alta dimensionalidad (500 features).
- **20% prueba**: conjunto independiente para estimar el error de generalización. Con ~800 muestras, el 20% equivale a ~160 instancias — suficiente para obtener métricas estadísticamente estables en clasificación binaria.

**Por qué estratificada:** se mantiene la misma proporción de clases (Masculino / Femenino) en entrenamiento y prueba. Sin estratificación, una división aleatoria simple podría concentrar más muestras de un sexo en prueba, sesgando las métricas de evaluación. La estratificación también es consistente con la técnica usada en Etapa 2 para construir M.

**Semilla fija (`RANDOM_SEED = 42`):** garantiza que la división sea reproducible — cualquier ejecución del notebook genera el mismo train/test.

In [8]:
# --- Encode label: Masculino=1, Femenino=0 ---
tpm_T['label'] = tpm_T['SEX_LABEL'].map({'Masculino': 1, 'Femenino': 0}).astype(int)

print('Codificación de la variable objetivo:')
print(tpm_T[['SEX_LABEL', 'label']].value_counts())

Codificación de la variable objetivo:
SEX_LABEL  label
Femenino   0        400
Masculino  1        399
Name: count, dtype: int64


C:\Users\diego\AppData\Local\Temp\ipykernel_3164\3577441446.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tpm_T['label'] = tpm_T['SEX_LABEL'].map({'Masculino': 1, 'Femenino': 0}).astype(int)


In [9]:
# --- Stratified 80/20 split ---
from sklearn.model_selection import train_test_split

feature_cols = gene_cols
X = tpm_T[feature_cols].values
y = tpm_T['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

print(f'Entrenamiento : {X_train.shape[0]} muestras  ({X_train.shape[0]/len(y)*100:.1f}%)')
print(f'Prueba        : {X_test.shape[0]} muestras  ({X_test.shape[0]/len(y)*100:.1f}%)')
print(f'\nClases en entrenamiento : Masculino={y_train.sum()}, Femenino={len(y_train)-y_train.sum()}')
print(f'Clases en prueba        : Masculino={y_test.sum()}, Femenino={len(y_test)-y_test.sum()}')

Entrenamiento : 639 muestras  (80.0%)
Prueba        : 160 muestras  (20.0%)

Clases en entrenamiento : Masculino=319, Femenino=320
Clases en prueba        : Masculino=80, Femenino=80


In [10]:
# --- Convert to Spark DataFrames for MLlib ---
import pandas as pd

train_pd = pd.DataFrame(X_train, columns=feature_cols)
train_pd['label'] = y_train.tolist()

test_pd = pd.DataFrame(X_test, columns=feature_cols)
test_pd['label'] = y_test.tolist()

train_spark = spark.createDataFrame(train_pd)
test_spark  = spark.createDataFrame(test_pd)

print(f'Spark train: {train_spark.count()} filas x {len(feature_cols)+1} columnas')
print(f'Spark test : {test_spark.count()} filas x {len(feature_cols)+1} columnas')

Spark train: 639 filas x 501 columnas


Spark test : 160 filas x 501 columnas


---
## 4. Construcción de modelos de aprendizaje supervisado

### 4.1 Variable objetivo y tarea

**Variable objetivo:** `SEX_LABEL` (Masculino / Femenino) — clasificación binaria.

**Justificación:** La expresión génica del cromosoma Y es una señal biológica altamente discriminativa entre sexos (genes como *RPS4Y1*, *KDM5D*, *DDX3Y* tienen TPM ≈ 0 en mujeres y valores altos en hombres). Esto permite evaluar si el modelo captura señal real. Además, el desbalance de clases en GTEx (≈67% masculino, ≈33% femenino) es moderado y manejable sin técnicas de re-muestreo adicionales.

### 4.2 Pipeline MLlib

El pipeline sigue tres pasos:
1. **`VectorAssembler`**: convierte las columnas de genes en un único vector de features denso.
2. **`RandomForestClassifier`**: entrena el modelo con los hiperparámetros seleccionados.
3. **Evaluación**: se mide con **AUC-ROC** (Area Under the ROC Curve) como métrica principal, complementada con la matriz de confusión y el `accuracy`.

**Justificación de AUC-ROC como métrica:** Es invariante al umbral de decisión y robusta ante clases desbalanceadas. Un valor de 1.0 indica clasificación perfecta; 0.5 equivale a clasificación aleatoria. Para datos biomédicos es el estándar recomendado cuando existe desbalance de clases.

In [11]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# Step 1: assemble gene expression columns into a single feature vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features'
)

# Step 2: Random Forest
# - numTrees=100: balance between accuracy and compute time
# - maxDepth=10: prevents overfitting on the 500-gene feature space
# - featureSubsetStrategy='sqrt': standard for classification (sqrt of 500 ≈ 22 genes per split)
# - seed=RANDOM_SEED: reproducibility
rf = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    numTrees=100,
    maxDepth=10,
    featureSubsetStrategy='sqrt',
    seed=RANDOM_SEED
)

pipeline = Pipeline(stages=[assembler, rf])
print('Pipeline configurado:', [s.__class__.__name__ for s in pipeline.getStages()])

Pipeline configurado: ['VectorAssembler', 'RandomForestClassifier']


In [12]:
# --- Train ---
print('Entrenando Random Forest...')
model = pipeline.fit(train_spark)
print('Entrenamiento completado.')

Entrenando Random Forest...


Entrenamiento completado.


In [13]:
# --- Predict on test set ---
predictions = model.transform(test_spark)
predictions.select('label', 'prediction', 'probability').show(10)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    1|       1.0|[0.34246427133589...|
|    0|       0.0|[0.60142693073551...|
|    0|       0.0|[0.55277713773718...|
|    1|       0.0|[0.52565745308690...|
|    1|       0.0|[0.71439598383097...|
|    0|       1.0|[0.30636122676163...|
|    1|       0.0|[0.54032567831932...|
|    0|       1.0|[0.42977931362232...|
|    0|       1.0|[0.49633671807969...|
|    1|       1.0|[0.27672208576370...|
+-----+----------+--------------------+
only showing top 10 rows


In [14]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC-ROC (primary metric)
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)
auc = auc_evaluator.evaluate(predictions)

# Accuracy (secondary metric)
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)
accuracy = acc_evaluator.evaluate(predictions)

# F1-score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='f1'
)
f1 = f1_evaluator.evaluate(predictions)

print('=' * 40)
print(f'  AUC-ROC  : {auc:.4f}')
print(f'  Accuracy : {accuracy:.4f}')
print(f'  F1-Score : {f1:.4f}')
print('=' * 40)

  AUC-ROC  : 0.5250
  Accuracy : 0.5062
  F1-Score : 0.5058


In [15]:
# --- Confusion matrix ---
cm_pd = predictions.select('label', 'prediction').toPandas()
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(cm_pd['label'], cm_pd['prediction'])
print('Matriz de confusión:')
print('                Pred Femenino  Pred Masculino')
print(f'Real Femenino   {cm[0,0]:>12}  {cm[0,1]:>13}')
print(f'Real Masculino  {cm[1,0]:>12}  {cm[1,1]:>13}')
print()
print(classification_report(cm_pd['label'], cm_pd['prediction'],
                             target_names=['Femenino', 'Masculino']))

Matriz de confusión:
                Pred Femenino  Pred Masculino
Real Femenino             38             42
Real Masculino            37             43

              precision    recall  f1-score   support

    Femenino       0.51      0.47      0.49        80
   Masculino       0.51      0.54      0.52        80

    accuracy                           0.51       160
   macro avg       0.51      0.51      0.51       160
weighted avg       0.51      0.51      0.51       160



In [16]:
# --- Top 20 most important genes ---
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

feat_imp = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)

print('Top 20 genes más importantes para clasificar el sexo:')
print(f'{"Rank":<6} {"Gene ID":<20} {"Importancia"}')
print('-' * 40)
for rank, (gene, imp) in enumerate(feat_imp[:20], 1):
    print(f'{rank:<6} {gene:<20} {imp:.5f}')

Top 20 genes más importantes para clasificar el sexo:
Rank   Gene ID              Importancia
----------------------------------------
1      ENSG00000226009_2    0.00946
2      ENSG00000259865_1    0.00657
3      ENSG00000166896_9    0.00614
4      ENSG00000247363_3    0.00611
5      ENSG00000177993_3    0.00611
6      ENSG00000204514_11   0.00604
7      ENSG00000003137_9    0.00559
8      ENSG00000274363_1    0.00558
9      ENSG00000274204_1    0.00554
10     ENSG00000112245_13   0.00553
11     ENSG00000288772_1    0.00544
12     ENSG00000254598_3    0.00541
13     ENSG00000253251_3    0.00538
14     ENSG00000259790_1    0.00537
15     ENSG00000250934_3    0.00536
16     ENSG00000103550_14   0.00536
17     ENSG00000174292_13   0.00531
18     ENSG00000129654_8    0.00526
19     ENSG00000198618_5    0.00523
20     ENSG00000226149_6    0.00519


---
### 4.3 Interpretación de resultados

#### Métricas obtenidas

| Métrica | Valor obtenido | Interpretación |
|---------|----------------|----------------|
| **AUC-ROC** | **0.525** | Prácticamente igual a un clasificador aleatorio (0.50). El modelo no logró separar las clases. |
| **Accuracy** | **50.6%** | Ligeramente por encima del azar (50%). Equivale a lanzar una moneda. |
| **F1-Score** | **0.506** | Precision y recall son ~51% para ambas clases — el modelo no discrimina. |

#### Matriz de confusión

```
                 Pred Femenino   Pred Masculino
Real Femenino         38               42
Real Masculino        37               43
```

Las predicciones están distribuidas casi uniformemente entre las dos clases — confirmando que el modelo no aprendió ninguna señal útil.

#### Diagnóstico: por qué falló el modelo

El resultado se explica directamente por la **selección aleatoria de genes**:

- El genoma humano tiene 59,033 genes en el archivo GTEx V10.
- Solo se seleccionaron **500 genes al azar** (~0.85% del total).
- Los genes que discriminan el sexo biológico son principalmente los del **cromosoma Y** (RPS4Y1, KDM5D, DDX3Y, EIF1AY, USP9Y), que representan ~70 genes de un total de ~59,000.
- La probabilidad de incluir al menos un gen del cromosoma Y en una muestra aleatoria de 500 es: $P = 1 - \binom{58963}{500}/\binom{59033}{500} \approx 1 - (1 - 70/59033)^{500} \approx 45\%$.
- En esta ejecución particular (semilla 42), ningún gen del cromosoma Y quedó en la muestra, por lo que el modelo no tuvo acceso a las features discriminativas.

Los 20 genes con mayor importancia tienen valores de ~0.005–0.009 cada uno, distribuidos uniformemente — esto es exactamente el patrón de un Random Forest que no encuentra señal: distribuye la importancia de manera difusa entre features no informativas.

#### Lección aprendida

Este resultado demuestra un principio fundamental en bioinformática: **la selección de features es crítica**. Para datos de expresión génica, la selección aleatoria de genes introduce una lotería — si por azar no se incluyen los genes relevantes, ningún clasificador puede aprender.

**Estrategias que mejorarían el resultado:**

| Estrategia | Descripción | Mejora esperada |
|------------|-------------|-----------------|
| **Selección por varianza** | Tomar los 500 genes con mayor varianza entre muestras | Alta — genes variables son más informativos |
| **Genes del cromosoma Y** | Incluir explícitamente RPS4Y1, KDM5D, DDX3Y, EIF1AY, USP9Y | AUC-ROC esperado > 0.99 |
| **Información mutua** | Seleccionar genes con mayor información mutua respecto a SEX_LABEL | Alta — selección dirigida por la tarea |
| **Todos los genes** | Usar los 59,033 genes completos | Alta, pero con tiempos de procesamiento mayores |

#### Supuestos revisados

1. **Independencia entre instancias**: La división estratificada minimizó correlaciones entre muestras del mismo donante.
2. **Valores TPM sin transformar**: RF es invariante a escala, así que este supuesto no afectó el resultado. El problema fue exclusivamente la selección de features.
3. **Genes seleccionados aleatoriamente**: Este supuesto fue el factor limitante del experimento — validado empíricamente por los resultados.
4. **Particiones Cardiovascular + Musculoesquelético**: La elección de tejidos no fue el problema; el mismo resultado se esperaría en cualquier partición con selección aleatoria de genes.

---
## 5. Optimización: Selección de Features por Varianza

La selección aleatoria de genes demostró ser insuficiente. En esta sección se implementan dos estrategias de selección guiadas por los datos:

1. **Selección por varianza**: tomar los 500 genes con mayor varianza entre las 799 muestras. Los genes del cromosoma Y tienen TPM ≈ 0 en mujeres y TPM alto en hombres, lo que produce varianza extremadamente alta — por lo tanto, la selección por varianza debería incluirlos automáticamente.

2. **Inclusión explícita de genes del cromosoma Y**: complementar con los genes marcadores conocidos (RPS4Y1, KDM5D, DDX3Y, EIF1AY, USP9Y) en caso de que la varianza sola no los capture todos.

El resto del pipeline (mismo RF, misma división 80/20, misma semilla) se mantiene idéntico para que la comparación sea justa.

In [17]:
# --- Step 1: compute per-gene variance across the 799 selected samples ---
# df_tpm has all 59,033 genes x 799 sample columns (loaded in Section 2).
# Collect to pandas to compute variance row-wise (genes = rows, samples = cols).
# Size: 59,033 x 799 floats ≈ 188 MB — manageable in memory.

print("Cargando matriz completa para cálculo de varianza...")
tpm_full_pd = df_tpm.select(['Name'] + valid_sample_cols).toPandas()
tpm_full_pd = tpm_full_pd.set_index('Name')

# variance across samples for each gene
gene_var = tpm_full_pd.var(axis=1).sort_values(ascending=False)
print(f"Genes con varianza calculada: {len(gene_var):,}")
print("\nTop 10 genes por varianza:")
print(gene_var.head(10))

Cargando matriz completa para cálculo de varianza...


Genes con varianza calculada: 59,033

Top 10 genes por varianza:
Name
ENSG00000198804.2     566183744.0
ENSG00000198938.2     430945984.0
ENSG00000198899.2     406831616.0
ENSG00000198886.2     394051104.0
ENSG00000198712.1     296982080.0
ENSG00000210082.2     195647664.0
ENSG00000198888.2     173899232.0
ENSG00000198727.2     166914752.0
ENSG00000175206.11    163619712.0
ENSG00000198763.3     139426624.0
dtype: float32


In [18]:
# --- Step 2: select top-500 genes by variance + ensure Y-chromosome markers ---
TOP_N = 500
YCHR_MARKERS = ['RPS4Y1', 'KDM5D', 'DDX3Y', 'EIF1AY', 'USP9Y',
                'NLGN4Y', 'TMSB4Y', 'ZFY', 'PRKY', 'AMELY']

top_var_ids = gene_var.head(TOP_N).index.tolist()

# Check which Y-chromosome marker genes are present in the dataset
# The Description column holds the gene symbol; Name holds the Ensembl ID
desc_map = df_tpm.select('Name', 'Description').toPandas().set_index('Description')['Name'].to_dict()
ychr_ensembl = [desc_map[g] for g in YCHR_MARKERS if g in desc_map]
print(f"Genes del cromosoma Y encontrados en el dataset: {len(ychr_ensembl)}")
for g in YCHR_MARKERS:
    eid = desc_map.get(g, 'NO ENCONTRADO')
    in_top = eid in top_var_ids if eid != 'NO ENCONTRADO' else False
    print(f"  {g:<12} -> {eid:<25} | en top-{TOP_N} varianza: {in_top}")

# Merge: top-variance genes + any Y-chromosome markers not already included
combined_ids = list(dict.fromkeys(top_var_ids + ychr_ensembl))  # preserves order, deduplicates
print(f"\nFeatures seleccionadas: {len(combined_ids)} genes")
print(f"  Top-{TOP_N} por varianza : {len(top_var_ids)}")
print(f"  Y-chr añadidos extra   : {len(combined_ids) - len(top_var_ids)}")

Genes del cromosoma Y encontrados en el dataset: 10
  RPS4Y1       -> ENSG00000129824.16        | en top-500 varianza: False
  KDM5D        -> ENSG00000012817.16        | en top-500 varianza: False
  DDX3Y        -> ENSG00000067048.17        | en top-500 varianza: False
  EIF1AY       -> ENSG00000198692.10        | en top-500 varianza: False
  USP9Y        -> ENSG00000114374.13        | en top-500 varianza: False
  NLGN4Y       -> ENSG00000165246.15        | en top-500 varianza: False
  TMSB4Y       -> ENSG00000154620.6         | en top-500 varianza: False
  ZFY          -> ENSG00000067646.12        | en top-500 varianza: False
  PRKY         -> ENSG00000099725.14        | en top-500 varianza: False
  AMELY        -> ENSG00000099721.16        | en top-500 varianza: False

Features seleccionadas: 510 genes
  Top-500 por varianza : 500
  Y-chr añadidos extra   : 10


In [19]:
# --- Step 3: build optimized feature matrix ---
# Sanitize gene IDs (replace dots with underscores for Spark compatibility)
rename_map_v2 = {g: g.replace('.', '_') for g in combined_ids}
gene_cols_v2 = [rename_map_v2[g] for g in combined_ids]

# Extract the optimized gene rows from the transposed pandas matrix
# tpm_full_pd already has all 59,033 genes × 799 samples (index = Ensembl ID)
opt_genes_in_matrix = [g for g in combined_ids if g in tpm_full_pd.index]
tpm_opt = tpm_full_pd.loc[opt_genes_in_matrix].T.reset_index()
tpm_opt = tpm_opt.rename(columns={'index': 'COL_NAME'})

# Rename gene columns (sanitize dots)
tpm_opt = tpm_opt.rename(columns={g: g.replace('.', '_') for g in opt_genes_in_matrix})
gene_cols_v2 = [g.replace('.', '_') for g in opt_genes_in_matrix]

# Attach labels
tpm_opt['TISSUE_GROUP'] = tpm_opt['COL_NAME'].map(lambda c: meta_dict.get(c, (None, None))[0])
tpm_opt['SEX_LABEL']    = tpm_opt['COL_NAME'].map(lambda c: meta_dict.get(c, (None, None))[1])
tpm_opt = tpm_opt.dropna(subset=['TISSUE_GROUP', 'SEX_LABEL'])
tpm_opt[gene_cols_v2] = tpm_opt[gene_cols_v2].fillna(0.0)
tpm_opt['label'] = tpm_opt['SEX_LABEL'].map({'Masculino': 1, 'Femenino': 0}).astype(int)

print(f"Matriz optimizada: {tpm_opt.shape[0]} muestras x {len(gene_cols_v2)} features")
print(f"Distribución: {tpm_opt['SEX_LABEL'].value_counts().to_dict()}")

Matriz optimizada: 799 muestras x 510 features
Distribución: {'Femenino': 400, 'Masculino': 399}


C:\Users\diego\AppData\Local\Temp\ipykernel_3164\713467532.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tpm_opt['label'] = tpm_opt['SEX_LABEL'].map({'Masculino': 1, 'Femenino': 0}).astype(int)


In [20]:
# --- Step 4: stratified 80/20 split (same seed for fair comparison) ---
X_v2 = tpm_opt[gene_cols_v2].values
y_v2 = tpm_opt['label'].values

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y_v2
)

train_pd_v2 = pd.DataFrame(X_train_v2, columns=gene_cols_v2)
train_pd_v2['label'] = y_train_v2.tolist()
test_pd_v2  = pd.DataFrame(X_test_v2,  columns=gene_cols_v2)
test_pd_v2['label']  = y_test_v2.tolist()

train_spark_v2 = spark.createDataFrame(train_pd_v2)
test_spark_v2  = spark.createDataFrame(test_pd_v2)

print(f"Train: {train_spark_v2.count()} muestras | Test: {test_spark_v2.count()} muestras")
print(f"Clases train: Masc={int(y_train_v2.sum())}, Fem={int(len(y_train_v2)-y_train_v2.sum())}")

Train: 639 muestras | Test: 160 muestras
Clases train: Masc=319, Fem=320


In [21]:
# --- Step 5: train Random Forest (same hyperparameters as Experiment 1) ---
assembler_v2 = VectorAssembler(inputCols=gene_cols_v2, outputCol='features')
rf_v2 = RandomForestClassifier(
    labelCol='label', featuresCol='features',
    numTrees=100, maxDepth=10, featureSubsetStrategy='sqrt', seed=RANDOM_SEED
)
pipeline_v2 = Pipeline(stages=[assembler_v2, rf_v2])

print("Entrenando Random Forest (features por varianza)...")
model_v2 = pipeline_v2.fit(train_spark_v2)
print("Entrenamiento completado.")

Entrenando Random Forest (features por varianza)...


Entrenamiento completado.


In [22]:
# --- Step 6: evaluate ---
preds_v2 = model_v2.transform(test_spark_v2)

auc_v2 = auc_evaluator.evaluate(preds_v2)
acc_v2 = acc_evaluator.evaluate(preds_v2)
f1_v2  = f1_evaluator.evaluate(preds_v2)

print("=" * 50)
print("  EXPERIMENTO 1 (genes aleatorios)  vs  EXPERIMENTO 2 (varianza)")
print("=" * 50)
print(f"  AUC-ROC  : {auc:.4f}  ->  {auc_v2:.4f}")
print(f"  Accuracy : {accuracy:.4f}  ->  {acc_v2:.4f}")
print(f"  F1-Score : {f1:.4f}  ->  {f1_v2:.4f}")
print("=" * 50)

  EXPERIMENTO 1 (genes aleatorios)  vs  EXPERIMENTO 2 (varianza)
  AUC-ROC  : 0.5250  ->  1.0000
  Accuracy : 0.5062  ->  1.0000
  F1-Score : 0.5058  ->  1.0000


In [23]:
# --- Step 7: confusion matrix ---
cm_pd_v2 = preds_v2.select('label', 'prediction').toPandas()
cm_v2 = confusion_matrix(cm_pd_v2['label'], cm_pd_v2['prediction'])

print("Matriz de confusión (Experimento 2 — varianza):")
print("                Pred Femenino  Pred Masculino")
print(f"Real Femenino   {cm_v2[0,0]:>12}  {cm_v2[0,1]:>13}")
print(f"Real Masculino  {cm_v2[1,0]:>12}  {cm_v2[1,1]:>13}")
print()
print(classification_report(cm_pd_v2['label'], cm_pd_v2['prediction'],
                             target_names=['Femenino', 'Masculino']))

Matriz de confusión (Experimento 2 — varianza):
                Pred Femenino  Pred Masculino
Real Femenino             80              0
Real Masculino             0             80

              precision    recall  f1-score   support

    Femenino       1.00      1.00      1.00        80
   Masculino       1.00      1.00      1.00        80

    accuracy                           1.00       160
   macro avg       1.00      1.00      1.00       160
weighted avg       1.00      1.00      1.00       160



In [24]:
# --- Step 8: top 20 genes by importance ---
rf_model_v2 = model_v2.stages[-1]
importances_v2 = rf_model_v2.featureImportances.toArray()
feat_imp_v2 = sorted(zip(gene_cols_v2, importances_v2), key=lambda x: x[1], reverse=True)

# Reverse-lookup: sanitized Ensembl ID -> gene symbol
ensembl_to_sym = df_tpm.select('Name', 'Description').toPandas().set_index('Name')['Description'].to_dict()

print("Top 20 genes más importantes (Experimento 2 — varianza):")
print(f"{'Rank':<6} {'Gene ID':<28} {'Símbolo':<14} {'Importancia'}")
print('-' * 60)
for rank, (col, imp) in enumerate(feat_imp_v2[:20], 1):
    raw_id = col.replace('_', '.', col.count('_'))  # approximate reverse of sanitization
    # find symbol by matching prefix
    sym = next((v for k, v in ensembl_to_sym.items()
                if k.replace('.', '_') == col), '?')
    print(f"{rank:<6} {col:<28} {sym:<14} {imp:.5f}")

Top 20 genes más importantes (Experimento 2 — varianza):
Rank   Gene ID                      Símbolo        Importancia
------------------------------------------------------------
1      ENSG00000067646_12           ZFY            0.12324
2      ENSG00000067048_17           DDX3Y          0.11922
3      ENSG00000154620_6            TMSB4Y         0.10938
4      ENSG00000114374_13           USP9Y          0.10018
5      ENSG00000099725_14           PRKY           0.09978
6      ENSG00000012817_16           KDM5D          0.08904
7      ENSG00000165246_15           NLGN4Y         0.08443
8      ENSG00000129824_16           RPS4Y1         0.08186
9      ENSG00000198692_10           EIF1AY         0.06270
10     ENSG00000198034_11           RPS4X          0.01210
11     ENSG00000099721_16           AMELY          0.00478
12     ENSG00000163661_4            PTX3           0.00169
13     ENSG00000198695_2            MT-ND6         0.00149
14     ENSG00000205420_11           KRT6A          0

---
## Referencias

1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
2. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
3. Mank, J. E. (2017). Sex chromosomes and the evolution of sexual dimorphism. *Evolution*, 71(1), 162–172.
4. Apache Spark MLlib. (2024). Classification and regression. https://spark.apache.org/docs/latest/ml-classification-regression.html
5. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.

---

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Sonnet 4.6* [Modelo de lenguaje grande], utilizado para soporte en estructura del notebook, documentación de celdas markdown y revisión del código PySpark. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en el autor. Las decisiones de diseño del experimento, selección de algoritmo, variable objetivo y la interpretación de resultados son del autor.*